# LCZ + Gridded Climate/Environmental Data — Integration Tutorial

This notebook demonstrates the `lcz_grid_*()` pipeline added to `lcz4r_python` (ported from climasus4r's `sus_grid_*()` R functions), and how to relate gridded climate/pollution/drought data to Local Climate Zones.

**Pipeline:** Load grid (municipalities) -> Extract/associate (Koppen, precipitation, ...) -> Join to a base dataset -> Cross with LCZ -> Visualize.

**Runs fully offline.** Functions that need a live download or authentication (NASA Earthdata for MERRA-2, a FIRMS MAP KEY, the ~3GB GHAP daily ZIPs) are demonstrated with small synthetic fixtures instead of real downloads, so every cell below executes without credentials or a large download. A few cells optionally hit real, confirmed-reachable endpoints (TerraBrasilis WFS, GHAP annual O3 on Zenodo) — these are guarded by a network check and skip cleanly if offline.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import box

import matplotlib
matplotlib.use('Agg')  # headless-safe; drop this line to render inline

# Quick, best-effort network check used to skip the two optional live cells below.
import httpx
try:
    httpx.get('https://zenodo.org', timeout=3.0)
    HAS_NETWORK = True
except Exception:
    HAS_NETWORK = False
print('Network available:', HAS_NETWORK)

Network available: True


## Step 1 — Synthetic municipalities

Three toy municipalities in Mato Grosso (Sinop, Cuiaba, Rondonopolis-ish bounding boxes), standing in for `geobr::read_municipality()` output in the R tutorials — `geobr` has no direct Python equivalent bundled here, so any `gpd.GeoDataFrame` with a `code_muni` column and polygon geometries works.

In [2]:
municipalities = gpd.GeoDataFrame(
    {
        'code_muni': ['5107602', '5103403', '5106190'],
        'name_muni': ['Sinop', 'Cuiaba', 'Rondonopolis'],
    },
    geometry=[
        box(-55.7, -11.9, -55.4, -11.6),
        box(-56.2, -15.7, -55.9, -15.4),
        box(-54.7, -16.6, -54.4, -16.3),
    ],
    crs=4326,
)
municipalities

,code_muni,name_muni,geometry
0,5107602,Sinop,"POLYGON ((-55.4 -11.9, -55.4 -11.6, -55.7 -11...."
1,5103403,Cuiaba,"POLYGON ((-55.9 -15.7, -55.9 -15.4, -56.2 -15...."
2,5106190,Rondonopolis,"POLYGON ((-54.4 -16.6, -54.4 -16.3, -54.7 -16...."


## Step 2 — Koppen-Geiger climate classification

`lcz_grid_koppen()` (`approx` mode) derives each municipality's centroid lon/lat directly from its geometry and classifies it — no download required.

In [3]:
from lcz_grid_koppen import lcz_grid_koppen

koppen = lcz_grid_koppen(municipalities, mode='approx', lang='en')
koppen[['code_muni', 'name_muni', 'zona_koppen']]

Assigning Koppen zones (mode: approx)...
Done. Distribution: Aw=3, Af=0, Am=0, As=0, BSh=0, Cf=0, Cw=0


/Users/co2map/Documents/lcz4r_python/lcz_grid_koppen.py:128: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = df.geometry.to_crs(4326).centroid


,code_muni,name_muni,zona_koppen
0,5107602,Sinop,Aw
1,5103403,Cuiaba,Aw
2,5106190,Rondonopolis,Aw


## Step 3 — Precipitation (CHIRPS)

`lcz_grid_chirps()` downloads and aggregates real CHIRPS GeoTIFFs. To keep this notebook runnable offline, we fabricate a small CHIRPS-shaped raster and place it directly in the cache directory that `lcz_grid_chirps()` would have downloaded to — the function then reads and aggregates it exactly as it would a real file (this is the same mechanism used in `test_lcz_grid.py`). Set `municipalities=None` and drop `cache_dir` to fetch the real CHIRPS archive over the network instead.

In [4]:
import gzip, shutil

chirps_cache = os.path.abspath('./_tutorial_cache/chirps')
os.makedirs(os.path.join(chirps_cache, 'monthly'), exist_ok=True)

transform = from_origin(-58, -10, 0.05, 0.05)
rng = np.random.default_rng(42)
rainfall = rng.uniform(0, 250, size=(200, 200))
plain_path = os.path.join(chirps_cache, 'monthly', 'chirps-v2.0.2022.01.tif')
with rasterio.open(plain_path, 'w', driver='GTiff', height=200, width=200, count=1,
                    dtype='float64', crs='EPSG:4326', transform=transform) as dst:
    dst.write(rainfall, 1)
with open(plain_path, 'rb') as f_in, gzip.open(plain_path + '.gz', 'wb') as f_out:
    shutil.copyfileobj(f_in, f_out)
os.remove(plain_path)

from lcz_grid_chirps import lcz_grid_chirps

rainfall_df = lcz_grid_chirps(
    resolution='monthly', years=[2022], months=[1],
    municipalities=municipalities, crop_brazil=False,
    use_cache=True, cache_dir=chirps_cache, verbose=False,
)
rainfall_df

,code_muni,date,rainfall_chirps_mm
0,5103403,2022-01-01,133.578056
1,5106190,2022-01-01,158.343692
2,5107602,2022-01-01,125.487148


## Step 4 — Join gridded data to a base (health-style) dataset

`lcz_grid_join()` is the bridge between any `lcz_grid_*()` output and a health/base table keyed by `code_muni` (+ `date`), mirroring `sus_grid_join()`.

In [5]:
health_like = pd.DataFrame({
    'code_muni': ['5107602', '5103403', '5106190'],
    'date': pd.to_datetime(['2022-01-01'] * 3),
    'resp_hospitalizations': [12, 27, 9],
})

from lcz_grid_join import lcz_grid_join

combined = lcz_grid_join(health_like, rainfall_df, verbose=False)
combined

,code_muni,date,resp_hospitalizations,rainfall_chirps_mm
0,5107602,2022-01-01,12,125.487148
1,5103403,2022-01-01,27,133.578056
2,5106190,2022-01-01,9,158.343692


## Step 5 — LCZ classification and an environmental variable, at pixel level

The grid functions above aggregate to *municipalities*; LCZ classification in this repo (`lcz_get_map`, `lcz_cal_area`) works *per pixel* on a single city's raster. To relate the two, `plot_lcz_relationship()` samples a gridded variable raster by LCZ class in raster space (see the module docstring for why this — not a `code_muni` join — is the right model here).

We synthesize an LCZ raster and a temperature raster for one city (a real one would come from `lcz_get_map('Cuiaba')` + `lcz_grid_era5(..., municipalities=None)` clipped to the same extent).

In [6]:
city_transform = from_origin(-56.15, -15.4, 0.002, 0.002)
h, w = 80, 80
rng = np.random.default_rng(7)
lcz_classes = rng.choice(np.arange(1, 18), size=(h, w))

lcz_path = './_tutorial_cache/lcz_cuiaba.tif'
with rasterio.open(lcz_path, 'w', driver='GTiff', height=h, width=w, count=1,
                    dtype='float64', crs='EPSG:4326', transform=city_transform) as dst:
    dst.write(lcz_classes.astype('float64'), 1)

# Urban heat island signal: dense/compact LCZ classes (low IDs) run hotter.
temperature = 34 - lcz_classes * 0.7 + rng.normal(0, 1.2, size=(h, w))
temp_path = './_tutorial_cache/temperature_cuiaba.tif'
with rasterio.open(temp_path, 'w', driver='GTiff', height=h, width=w, count=1,
                    dtype='float64', crs='EPSG:4326', transform=city_transform) as dst:
    dst.write(temperature, 1)

print('Synthetic LCZ + temperature rasters written.')

Synthetic LCZ + temperature rasters written.


## Step 6 — `plot_lcz_relationship`: variable distribution by LCZ class

In [7]:
from plot_lcz_relationship import plot_lcz_relationship

fig_box = plot_lcz_relationship(lcz_path, temp_path, variable_name='Temperature (C)', plot_type='box')
fig_box.write_html('./_tutorial_cache/lcz_relationship_box.html')
print('Saved: ./_tutorial_cache/lcz_relationship_box.html')

fig_heat = plot_lcz_relationship(lcz_path, temp_path, variable_name='Temperature (C)', plot_type='heatmap')
fig_heat

Saved: ./_tutorial_cache/lcz_relationship_box.html


## Step 7 — `plot_grid_only`: the municipality grid over a basemap

Colors each municipality by its January 2022 CHIRPS rainfall. Requires network access to fetch OpenStreetMap tiles — falls back to no basemap automatically if `contextily` isn't installed, and we skip the tile fetch here if the network check at the top of the notebook failed.

In [8]:
from plot_grid_only import plot_grid_only

muni_with_rain = municipalities.merge(rainfall_df[['code_muni', 'rainfall_chirps_mm']], on='code_muni')
fig = plot_grid_only(
    muni_with_rain, color_by='rainfall_chirps_mm',
    add_basemap=HAS_NETWORK, title='CHIRPS rainfall (mm) by municipality, Jan 2022',
)
fig.savefig('./_tutorial_cache/grid_rainfall_map.png', dpi=100, bbox_inches='tight')
print('Saved: ./_tutorial_cache/grid_rainfall_map.png')

Saved: ./_tutorial_cache/grid_rainfall_map.png


## Step 8 (optional, live) — Real PRODES deforestation and GHAP O3

These two calls hit real, confirmed-reachable public endpoints (TerraBrasilis WFS and a Zenodo GHAP file) with no authentication and no large downloads. They only run if the network check at the top of the notebook succeeded; otherwise this cell is skipped.

In [9]:
if HAS_NETWORK:
    from lcz_grid_prodes import lcz_grid_prodes
    from lcz_grid_pollution_ghap import lcz_grid_pollution_ghap

    prodes = lcz_grid_prodes(
        years=[2020], biomes=['Amazon'],
        municipalities=municipalities[municipalities.code_muni == '5107602'],  # Sinop
        use_cache=True, cache_dir='./_tutorial_cache/prodes', verbose=False,
    )
    print('PRODES deforestation (Sinop, 2020):')
    print(prodes)

    ghap_o3 = lcz_grid_pollution_ghap(
        pollutants=['o3'], resolution='annual', years=[2010],
        municipalities=municipalities[municipalities.code_muni == '5103403'],  # Cuiaba
        use_cache=True, cache_dir='./_tutorial_cache/ghap', verbose=False,
    )
    print('GHAP annual O3 (Cuiaba, 2010):')
    print(ghap_o3)
else:
    print('No network detected — skipping the live PRODES/GHAP demo.')

PRODES deforestation (Sinop, 2020):
  code_muni  year  deforested_area_km2  n_patches   biome       date
0   5107602  2020             3.155915          3  Amazon 2020-01-01


GHAP annual O3 (Cuiaba, 2010):
  code_muni       date    o3_mean
0   5103403 2010-01-01  30.308334


## Not runnable offline: authentication/large-download functions

The remaining `lcz_grid_*()` functions are code-complete and unit-tested (see `test_lcz_grid.py`) but can't run in a plain notebook cell:

- **`lcz_grid_pollution_merra2()`** — requires a free NASA Earthdata Login (`EARTHDATA_USER`/`EARTHDATA_PASSWORD` env vars, or a `.netrc` file).
- **`lcz_grid_fires(source='firms_modis'` or `'firms_viirs')`** — requires a free NASA FIRMS `MAP_KEY`. `source='inpe'` (the default) needs no key and works the same way as the CHIRPS/PRODES cells above.
- **`lcz_grid_pollution_ghap(resolution='daily')`** — downloads a ~3GB monthly ZIP per call; use `resolution='monthly'` or `'annual'` for anything but a dedicated batch job.
- **`lcz_grid_smvi()`** — downloads a ~109MB flash-drought event archive from HydroShare on first call.

Set the relevant credentials/env vars and swap in a real `municipalities` GeoDataFrame (e.g. from `geobr` via `rpy2`, or any polygon source with a municipality code column) to run them for real.

## Summary

This notebook exercised the full pipeline end to end: **grid download/aggregation** (Koppen, CHIRPS) -> **join to a base dataset** (`lcz_grid_join`) -> **LCZ x variable relationship** (`plot_lcz_relationship`) -> **spatial grid visualization** (`plot_grid_only`), plus two live calls against real public data sources.